# バイオ技術 1-5：論文再現チャレンジ

## 公開RNA-seqデータから、論文の重要遺伝子にたどり着けるか？

午前中は、RNA-seq解析の基本的な流れを学びました。

午後は、実際に論文で使われた公開RNA-seqデータを解析し、

> **自分たちの解析で、論文が注目した遺伝子にたどり着けるか？**

に挑戦します。

今回扱う研究は、ヒト気道平滑筋細胞にデキサメタゾン（Dex）を処理し、
遺伝子発現の変化をRNA-seqで調べた研究です。

---

## 今日のゴール

1. 論文の研究目的を理解する
2. count matrixとmetadataを確認する
3. 低発現遺伝子を除外する
4. PCAでサンプル間の関係を見る
5. paired designを意識してDEG解析を行う
6. Volcano plotとヒートマップを作る
7. 自分で候補遺伝子を3つ選ぶ
8. 論文が注目した遺伝子と比較する
9. 次に行うwet実験を考える

---

## 今日の流れ（約2時間）

| 時間 | 内容 |
|---|---|
| 10分 | 論文と研究背景 |
| 10分 | count matrixとmetadata |
| 10分 | 低発現遺伝子のfilter |
| 15分 | PCA |
| 25分 | DEG解析 |
| 15分 | Volcano plotとヒートマップ |
| 15分 | 候補遺伝子を3つ選ぶ |
| 10分 | 論文との答え合わせ |
| 10分 | 次のwet実験を考える |

---

## 大切な注意

このNotebookでは、Pythonだけで理解しやすく進めるため、

- library size補正
- log2(CPM + 1)
- paired t-test
- Benjamini-Hochberg補正

を使った**教育用の簡易解析**を行います。

論文結果を厳密に再現するための完全な解析ではありません。
本格的なRNA-seq解析では、DESeq2などのcount-based methodや、
実験デザインを考慮した統計モデルを使います。

今回の目的は、

> **公開データを自分で解析し、論文の結果の一部を手元で確かめる**

ことです。


---
# 0. 研究背景

喘息治療では、グルココルチコイドが広く使われています。

今回の研究では、4人のドナーから得られたヒト気道平滑筋細胞について、

- untreated：無処理
- Dex：dexamethasone処理

を比較しています。

各ドナーについて untreated と Dex の両方があるため、
**同じドナー内で処理前後を比較できるpaired design**になっています。

### 今日の研究上の問い

> **Dex処理によって、どの遺伝子の発現が変化するのか？**

そして最後に、

> **論文が注目した遺伝子を、私たちの解析でも見つけられるか？**

を確かめます。

### まだ論文の「答え」は見ないでください

まずは自分で解析し、自分の候補遺伝子を選びます。

論文の主要候補遺伝子は後半で明らかにします。


---
# 1. ライブラリを読み込む


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_rel
from sklearn.decomposition import PCA

from IPython.display import display, Markdown
from urllib.parse import quote

print("Libraries loaded.")


---
# 2. 公開データを読み込む

今回は、Bioconductorの`airway`データセットから書き出された、

- gene × sample のraw count matrix
- sample metadata

を読み込みます。

元データのGEO accessionは **GSE52778** です。

Notebookを実行すると、データをインターネットから直接読み込みます。


In [ ]:
COUNTS_URL = (
    "https://raw.githubusercontent.com/"
    "mandhri/airway-glucocorticoid-rnaseq/"
    "main/data/raw/airway_counts.tsv"
)

META_URL = (
    "https://raw.githubusercontent.com/"
    "mandhri/airway-glucocorticoid-rnaseq/"
    "main/data/raw/airway_coldata.tsv"
)

counts = pd.read_csv(
    COUNTS_URL,
    sep="\t",
    index_col=0
)

metadata = pd.read_csv(
    META_URL,
    sep="\t"
)

print("Count matrix shape:", counts.shape)
print("Metadata shape:", metadata.shape)


### Count matrixを確認する

- 行：gene
- 列：sample
- 値：read count

です。


In [ ]:
display(counts.head())


### Metadataを確認する

重要な列は、

- `sample`：RNA-seq sample ID（SRR accession）
- `cell`：donor / cell line
- `dex`：Dex処理の有無

です。

ただし、`SRR1039508`のようなIDだけでは条件がわかりにくいため、
このNotebookではグラフ表示用に、

```text
Donor_Untreated
Donor_Dex
```

という`sample_label`を作って使います。


In [ ]:
display(
    metadata[
        ["sample", "cell", "dex"]
    ]
)


### ミニ確認1

次の問いに答えてください。

1. サンプル数はいくつですか？
2. donorは何人ですか？
3. untreatedは何サンプルですか？
4. Dex処理は何サンプルですか？
5. なぜ`cell`列を無視してはいけないと思いますか？

答え：

1.
2.
3.
4.
5.


---
# 3. Metadataをわかりやすく整える

`dex`列では、

- `untrt` = untreated
- `trt` = Dex treated

となっています。

表示用にわかりやすいラベルを作ります。


In [ ]:
metadata["condition"] = metadata["dex"].map({
    "untrt": "Untreated",
    "trt": "Dex"
})

metadata["condition"] = pd.Categorical(
    metadata["condition"],
    categories=["Untreated", "Dex"],
    ordered=True
)

# グラフ表示用のわかりやすいラベル
metadata["sample_label"] = (
    metadata["cell"].astype(str)
    + "_"
    + metadata["condition"].astype(str)
)

display(
    metadata[
        [
            "sample",
            "cell",
            "condition",
            "sample_label"
        ]
    ]
)


---
# 4. まずデータの大きさを見る

各sampleで得られたread countの合計を確認します。


In [ ]:
library_sizes = counts.sum(axis=0)

library_size_df = pd.DataFrame({
    "sample": library_sizes.index,
    "total_counts": library_sizes.values
}).merge(
    metadata[
        [
            "sample",
            "cell",
            "condition",
            "sample_label"
        ]
    ],
    on="sample",
    how="left"
)

display(library_size_df)


In [ ]:
# 表示順を Dex → Untreated にそろえる
library_plot_order = (
    metadata.loc[
        metadata["condition"] == "Dex",
        "sample_label"
    ].tolist()
    +
    metadata.loc[
        metadata["condition"] == "Untreated",
        "sample_label"
    ].tolist()
)

plt.figure(figsize=(11, 5))

sns.barplot(
    data=library_size_df,
    x="sample_label",
    y="total_counts",
    hue="condition",
    order=library_plot_order
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Sample (Donor_Condition)")
plt.ylabel("Total counts")
plt.title("Library size of each sample")
plt.tight_layout()
plt.show()


### 考えてみよう

sampleごとの総read countは完全に同じでしょうか？

答え：

この違いがあるため、raw countをそのままsample間で比較するのは適切ではありません。
後でCPMを使った簡易的な補正を行います。


---
# 5. 低発現遺伝子を除外する

非常に低いcountしか持たない遺伝子は、

- 測定ノイズの影響を受けやすい
- 統計解析が不安定になりやすい
- 多重検定の対象を不必要に増やす

可能性があります。

今回は、

> **count >= MIN_COUNT を満たすsampleが、MIN_SAMPLES個以上ある**

遺伝子を残します。

まず自分で条件を確認してください。


In [ ]:
# ↓↓↓ 条件を変更して試すことができます ↓↓↓
MIN_COUNT = 10
MIN_SAMPLES = 2

keep = (
    (counts >= MIN_COUNT).sum(axis=1)
    >= MIN_SAMPLES
)

filtered_counts = counts.loc[keep].copy()

print("Before filtering:", counts.shape[0], "genes")
print("After filtering :", filtered_counts.shape[0], "genes")
print("Removed         :", counts.shape[0] - filtered_counts.shape[0], "genes")


### ミニ演習2：filter条件を変える

例えば、

- `MIN_COUNT = 5`
- `MIN_COUNT = 10`
- `MIN_COUNT = 20`

や、

- `MIN_SAMPLES = 2`
- `MIN_SAMPLES = 4`

を試してください。

| MIN_COUNT | MIN_SAMPLES | 残ったgene数 |
|---|---|---|
| | | |
| | | |
| | | |

### 考えてみよう

filterを厳しくすると、残るgene数はどうなりますか？

答え：


---
# 6. CPMとlog変換

raw countにはlibrary sizeの違いがあります。

ここでは簡単な補正として、

> Counts Per Million（CPM）

を計算します。

その後、

> log2(CPM + 1)

へ変換します。


In [ ]:
# library sizeはfilter前の全geneから計算した値を使う
cpm = filtered_counts.div(
    library_sizes,
    axis=1
) * 1_000_000

log_cpm = np.log2(
    cpm + 1
)

print("CPM:")
display(cpm.head())

print("\nlog2(CPM + 1):")
display(log_cpm.head())


---
# 7. PCAでsample全体の違いを見る

解析前に予想してください。

> UntreatedとDexはPCA上で分かれると思いますか？

予想：

理由：

---

今回は、変動の大きい上位500遺伝子を使ってPCAを行います。


In [ ]:
# geneごとの分散
gene_variance = log_cpm.var(axis=1)

# 分散上位500 gene
top_variable_genes = (
    gene_variance
    .sort_values(ascending=False)
    .head(500)
    .index
)

pca_input = (
    log_cpm
    .loc[top_variable_genes]
    .T
)

pca = PCA(n_components=2)

pca_scores = pca.fit_transform(
    pca_input
)

pca_df = pd.DataFrame({
    "sample": pca_input.index,
    "PC1": pca_scores[:, 0],
    "PC2": pca_scores[:, 1]
}).merge(
    metadata[
        [
            "sample",
            "cell",
            "condition",
            "sample_label"
        ]
    ],
    on="sample",
    how="left"
)

print(
    "Explained variance:",
    pca.explained_variance_ratio_
)

display(pca_df)


In [ ]:
plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="condition",
    style="cell",
    s=150
)

# 同じdonorのUntreatedとDexを線で結ぶ
for donor, donor_df in pca_df.groupby("cell"):
    donor_df = donor_df.sort_values("condition")
    plt.plot(
        donor_df["PC1"],
        donor_df["PC2"],
        linestyle="--",
        alpha=0.5
    )

# 各点に Donor_Condition を表示
for _, row in pca_df.iterrows():
    plt.text(
        row["PC1"],
        row["PC2"],
        row["sample_label"],
        fontsize=8,
        alpha=0.85
    )

plt.title("PCA of airway RNA-seq samples")
plt.tight_layout()
plt.show()


### ミニ演習3：PCAを解釈する

1. UntreatedとDexは分かれて見えますか？
2. donorの違いも見えますか？
3. 同じdonorについて、Dex処理による移動方向は似ていますか？
4. この実験がpaired designである意味を考えてください。

答え：

1.
2.
3.
4.


---
# 8. Paired designを確認する

この実験では、同じdonorについて、

- Untreated
- Dex

が1組になっています。

比較に使うsampleの対応を確認します。


In [ ]:
donor_order = metadata["cell"].drop_duplicates().tolist()

untreated_samples = []
dex_samples = []
untreated_labels = []
dex_labels = []

for donor in donor_order:
    donor_meta = metadata[
        metadata["cell"] == donor
    ]

    untreated_row = donor_meta.loc[
        donor_meta["condition"] == "Untreated"
    ].iloc[0]

    dex_row = donor_meta.loc[
        donor_meta["condition"] == "Dex"
    ].iloc[0]

    untreated_samples.append(
        untreated_row["sample"]
    )
    dex_samples.append(
        dex_row["sample"]
    )

    untreated_labels.append(
        untreated_row["sample_label"]
    )
    dex_labels.append(
        dex_row["sample_label"]
    )

pair_table = pd.DataFrame({
    "donor": donor_order,
    "Untreated sample": untreated_samples,
    "Untreated label": untreated_labels,
    "Dex sample": dex_samples,
    "Dex label": dex_labels
})

display(pair_table)


### 重要

この対応を保ったままpaired testを行います。

例えば、

```text
Donor A: Untreated ↔ Dex
Donor B: Untreated ↔ Dex
Donor C: Untreated ↔ Dex
Donor D: Untreated ↔ Dex
```

の対応を崩してはいけません。


---
# 9. DEG解析

ここでは、各geneについて、

1. Dex群の平均発現量
2. Untreated群の平均発現量
3. log2 fold change
4. paired t-test
5. Benjamini-Hochberg補正

を計算します。

### log2FCの向き

このNotebookでは、

> **Dex - Untreated**

です。

したがって、

- log2FC > 0：Dexで高い
- log2FC < 0：Untreatedで高い

と解釈します。


In [ ]:
# group mean
untreated_mean = (
    log_cpm[untreated_samples]
    .mean(axis=1)
)

dex_mean = (
    log_cpm[dex_samples]
    .mean(axis=1)
)

log2fc = (
    dex_mean
    - untreated_mean
)

# paired t-test
statistic, p_values = ttest_rel(
    log_cpm[dex_samples].to_numpy(),
    log_cpm[untreated_samples].to_numpy(),
    axis=1,
    nan_policy="omit"
)

p_values = np.asarray(
    p_values,
    dtype=float
)

p_values = np.where(
    np.isfinite(p_values),
    p_values,
    1.0
)

print("DE test complete.")


## Benjamini-Hochberg補正

RNA-seqでは、多数の遺伝子について同時に検定します。

そのため、p値をそのまま使うだけでは偶然のfalse positiveが増えます。

ここではBenjamini-Hochberg法で多重検定補正を行います。


In [ ]:
def benjamini_hochberg(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float
    )

    n = len(p_values)

    order = np.argsort(p_values)
    ranked_p = p_values[order]

    adjusted = (
        ranked_p
        * n
        / np.arange(1, n + 1)
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.clip(
        adjusted,
        0,
        1
    )

    q_values = np.empty(n)
    q_values[order] = adjusted

    return q_values


padj = benjamini_hochberg(
    p_values
)

results = pd.DataFrame({
    "ensembl_id": log_cpm.index,
    "untreated_mean": untreated_mean.values,
    "dex_mean": dex_mean.values,
    "log2FC": log2fc.values,
    "pvalue": p_values,
    "padj": padj
})

results["minus_log10_padj"] = -np.log10(
    results["padj"].clip(lower=1e-300)
)

display(results.head())


---
# 10. Gene symbolを付ける

RNA-seqのcount matrixでは、geneがEnsembl IDで表現されています。

候補遺伝子を調べやすくするため、HGNCの公開対応表を使ってgene symbolを付けます。

インターネット接続などによりHGNCファイルを取得できない場合でも、
解析はEnsembl IDのまま続けられます。


In [ ]:
HGNC_URL = (
    "https://storage.googleapis.com/"
    "public-download-files/hgnc/tsv/tsv/"
    "hgnc_complete_set.txt"
)

try:
    hgnc = pd.read_csv(
        HGNC_URL,
        sep="\t",
        low_memory=False,
        dtype=str
    )

    gene_map = (
        hgnc[
            ["ensembl_gene_id", "symbol"]
        ]
        .dropna()
        .drop_duplicates(
            subset="ensembl_gene_id"
        )
    )

    results = results.merge(
        gene_map,
        left_on="ensembl_id",
        right_on="ensembl_gene_id",
        how="left"
    )

    results = results.drop(
        columns=["ensembl_gene_id"]
    )

    print(
        "Mapped gene symbols:",
        results["symbol"].notna().sum()
    )

except Exception as e:
    print(
        "HGNC mapping could not be downloaded."
    )
    print(
        "Analysis will continue with Ensembl IDs."
    )
    print(
        "Error:",
        e
    )

    results["symbol"] = np.nan


# 念のためCRISPLD2の対応を補う
crispld2_id = "ENSG00000103196"

results.loc[
    results["ensembl_id"] == crispld2_id,
    "symbol"
] = "CRISPLD2"

results["gene"] = results["symbol"].fillna(
    results["ensembl_id"]
)

display(
    results[
        [
            "gene",
            "ensembl_id",
            "log2FC",
            "pvalue",
            "padj"
        ]
    ].head()
)


---
# 11. DEGの基準を決める

ここからは自分たちで候補を絞ります。

今回は最初の基準として、

- |log2FC| > 1
- adjusted p-value < 0.05

を使います。

値を書き換えて、候補数がどう変わるか試すこともできます。


In [ ]:
# ↓↓↓ 自分で変更できます ↓↓↓
LOG2FC_THRESHOLD = 1.0
PADJ_THRESHOLD = 0.05

results["is_DEG"] = (
    results["log2FC"].abs()
    > LOG2FC_THRESHOLD
) & (
    results["padj"]
    < PADJ_THRESHOLD
)

results["direction"] = "Not significant"

results.loc[
    results["is_DEG"]
    & (results["log2FC"] > 0),
    "direction"
] = "Dex high"

results.loc[
    results["is_DEG"]
    & (results["log2FC"] < 0),
    "direction"
] = "Untreated high"

print(
    "Total DEG:",
    int(results["is_DEG"].sum())
)

print(
    "Dex high:",
    int(
        (results["direction"] == "Dex high")
        .sum()
    )
)

print(
    "Untreated high:",
    int(
        (results["direction"] == "Untreated high")
        .sum()
    )
)


### ミニ演習4：閾値を変える

例えば、

- log2FC threshold = 0.5
- log2FC threshold = 1.0
- log2FC threshold = 2.0

を比較してください。

| log2FC threshold | DEG数 |
|---|---|
| 0.5 | |
| 1.0 | |
| 2.0 | |

### 考えてみよう

候補遺伝子が多すぎる場合、wet実験では何が困るでしょうか？

答え：

逆に厳しすぎる基準には、どんな問題がありそうですか？

答え：


---
# 12. Volcano plot

Volcano plotでは、

- 横軸：log2 fold change
- 縦軸：-log10(adjusted p-value)

を表示します。

右側はDexで高い遺伝子、
左側はUntreatedで高い遺伝子です。


In [ ]:
plt.figure(figsize=(10, 7))

colors = {
    "Not significant": "lightgray",
    "Dex high": "tomato",
    "Untreated high": "royalblue"
}

for group in [
    "Not significant",
    "Dex high",
    "Untreated high"
]:
    sub = results[
        results["direction"] == group
    ]

    plt.scatter(
        sub["log2FC"],
        sub["minus_log10_padj"],
        s=14,
        alpha=0.65,
        label=group,
        color=colors[group]
    )

plt.axvline(
    LOG2FC_THRESHOLD,
    linestyle="--"
)

plt.axvline(
    -LOG2FC_THRESHOLD,
    linestyle="--"
)

plt.axhline(
    -np.log10(PADJ_THRESHOLD),
    linestyle="--"
)

plt.xlabel("log2 fold change (Dex - Untreated)")
plt.ylabel("-log10(adjusted p-value)")
plt.title("Volcano Plot")
plt.legend()
plt.tight_layout()
plt.show()


### 考えてみよう

Volcano plotを見てください。

1. Dexで上昇する遺伝子は多そうですか？
2. Untreatedで高い遺伝子もありますか？
3. 右上・左上にある点はどのような遺伝子ですか？

答え：

1.
2.
3.


---
# 13. Dexで上昇した上位遺伝子を見る

今回の研究上の問いは、

> **Dex処理によって誘導される遺伝子を探す**

ことです。

そこで、候補遺伝子リストは、

- **log2FC > 2.0**
- **adjusted p-value < 0.05**
- **Dexで発現上昇した遺伝子のみ**

に絞ります。

その中から、adjusted p-valueの小さい順に**上位50遺伝子**を表示します。

まだ論文の主要遺伝子は強調しません。

> **この候補リストの中から、自分ならどの遺伝子を選ぶか？**

という視点で表を見てください。

候補を選ぶときは、p値だけでなく、

- log2FCの大きさ
- 4 donorでの一貫性
- 生物学的な機能
- 次のwet実験につなげやすいか

も考えてみましょう。


In [ ]:
TOP_LIST_LOG2FC = 2.0
TOP_LIST_PADJ = 0.05
TOP_N = 50

top_candidates = (
    results[
        (results["log2FC"] > TOP_LIST_LOG2FC)
        & (results["padj"] < TOP_LIST_PADJ)
    ]
    .sort_values(
        "padj",
        ascending=True
    )
    .head(TOP_N)
    .copy()
)

top_candidates["change"] = "Dex high"

print(
    f"Criteria: log2FC > {TOP_LIST_LOG2FC}, "
    f"adjusted p-value < {TOP_LIST_PADJ}"
)

print(
    f"Dex-induced candidate genes shown: "
    f"{len(top_candidates)} "
    f"(maximum {TOP_N})"
)

display(
    top_candidates[
        [
            "gene",
            "ensembl_id",
            "log2FC",
            "pvalue",
            "padj",
            "change"
        ]
    ].reset_index(drop=True)
)


---
# 14. Dex誘導候補遺伝子のヒートマップ

ここでは、前のセクションで作成した**Dex-induced candidate list**から、
adjusted p-valueの小さい順に上位50遺伝子を使います。

列の並び順は、

> **Dex 4 samples → Untreated 4 samples**

とします。

この並びにすることで、

- Dex群で一貫して高いか
- Untreated群との差が明確か
- donorによるばらつきがあるか

を見やすくします。

p値やlog2FCだけでなく、

> **4人のdonorで一貫した変化を示しているか**

にも注目してください。


In [ ]:
# Dex-induced candidate listから上位30遺伝子を使用
top_heatmap_genes = (
    top_candidates
    .head(50)
    .copy()
)

top_ids = top_heatmap_genes[
    "ensembl_id"
].tolist()

# 列の順番を Dex → Untreated にする
dex_order = metadata.loc[
    metadata["condition"] == "Dex",
    "sample"
].tolist()

untreated_order = metadata.loc[
    metadata["condition"] == "Untreated",
    "sample"
].tolist()

heatmap_sample_order = (
    dex_order
    + untreated_order
)

heatmap_data = log_cpm.loc[
    top_ids,
    heatmap_sample_order
].copy()

# 列名をSRR IDからDonor_Conditionへ変更
sample_label_map = dict(
    zip(
        metadata["sample"],
        metadata["sample_label"]
    )
)

heatmap_data = heatmap_data.rename(
    columns=sample_label_map
)

# geneごとにz-score
heatmap_z = heatmap_data.sub(
    heatmap_data.mean(axis=1),
    axis=0
).div(
    heatmap_data.std(axis=1).replace(
        0,
        np.nan
    ),
    axis=0
)

# Ensembl IDをgene symbolへ置換
gene_name_map = (
    top_heatmap_genes
    .set_index("ensembl_id")["gene"]
    .to_dict()
)

heatmap_z.index = [
    gene_name_map.get(
        gene_id,
        gene_id
    )
    for gene_id in top_ids
]

plt.figure(figsize=(12, 10))

sns.heatmap(
    heatmap_z,
    cmap="vlag",
    center=0
)

# DexとUntreatedの境界線
plt.axvline(
    x=len(dex_order),
    color="black",
    linewidth=2
)

plt.xlabel("Sample (Dex → Untreated)")
plt.ylabel("Gene")
plt.title(
    "Top 30 Dex-Induced Candidate Genes"
)

plt.tight_layout()
plt.show()


### ミニ演習5：ヒートマップを読む

ヒートマップでは、

- 左：Dex 4 samples
- 右：Untreated 4 samples

です。

次の点を確認してください。

1. Dex群とUntreated群で発現パターンが分かれていますか？
2. Dex群の4 samplesで一貫して高いgeneはありますか？
3. donor間のばらつきが大きそうなgeneはありますか？
4. 候補を選ぶとき、p値だけでなくヒートマップを見る意味は何だと思いますか？

答え：

1.
2.
3.
4.


---
# 15. 自分の候補遺伝子を3つ選ぶ

ここが今日の中心です。

次の3つを参考に、

- **Dex-induced candidate list（上位50）**
- **Volcano plot**
- **Dex-induced candidatesの上位30遺伝子ヒートマップ**

> **次にwet実験で調べたい遺伝子を3つ選んでください**

選ぶ基準は1つではありません。

例えば、

- log2FCが大きい
- adjusted p-valueが小さい
- 4人のdonorで一貫して上昇している
- 発現量そのものが十分にある
- 機能が面白そう
- Dex応答との関係が興味深そう
- 自分の研究テーマに関係しそう

などです。

CRISPLD2は候補リストのかなり上位ではありますが、
「最上位だから選ばれた」というわけではありません。

研究では、

> **統計的順位 + 生物学的な意味 + 次の実験につながる仮説**

を組み合わせて候補を選ぶことが重要です。

### 私の候補

#### Candidate 1
- Gene：
- 選んだ理由：

#### Candidate 2
- Gene：
- 選んだ理由：

#### Candidate 3
- Gene：
- 選んだ理由：


### 選んだgeneを入力する

上の表に表示されているgene symbolまたはEnsembl IDを入力してください。

HGNC mappingが成功していればgene symbolを使えます。


In [ ]:
# ↓↓↓ 自分で3つ入力してください ↓↓↓
CANDIDATE_GENES = [
    "",
    "",
    ""
]

candidate_results = results[
    results["gene"].isin(CANDIDATE_GENES)
    | results["ensembl_id"].isin(CANDIDATE_GENES)
]

display(
    candidate_results[
        [
            "gene",
            "ensembl_id",
            "log2FC",
            "pvalue",
            "padj",
            "direction"
        ]
    ]
)


---
# 16. 遺伝子の機能を調べる

候補遺伝子について、

- NCBI Gene
- UniProt
- Ensembl
- PubMed

などで機能を調べてみましょう。

下のセルでは、候補遺伝子の検索リンクを表示します。


In [ ]:
for gene in CANDIDATE_GENES:
    if gene.strip() == "":
        continue

    q = quote(
        f"{gene} human gene"
    )

    display(
        Markdown(
            f"### {gene}\n"
            f"- [NCBI search](https://www.ncbi.nlm.nih.gov/search/all/?term={q})\n"
            f"- [UniProt search](https://www.uniprot.org/uniprotkb?query={q})\n"
            f"- [PubMed search](https://pubmed.ncbi.nlm.nih.gov/?term={q})"
        )
    )


### 調べた内容を記録する

#### Candidate 1
- Gene：
- 主な機能：
- この実験との関係として考えられること：

#### Candidate 2
- Gene：
- 主な機能：
- この実験との関係として考えられること：

#### Candidate 3
- Gene：
- 主な機能：
- この実験との関係として考えられること：


---
# 17. 論文との答え合わせ

ここまで来たら、論文が注目した主要候補遺伝子を確認します。

## 論文が注目した遺伝子

# CRISPLD2

Himes et al.は、Dexで発現が上昇した遺伝子の中からCRISPLD2に注目し、
その後の実験的検証を行いました。

では、私たちの簡易解析でもCRISPLD2は候補リストに入っていたでしょうか？


In [ ]:
CRISPLD2_ID = "ENSG00000103196"

crispld2_result = results[
    results["ensembl_id"]
    == CRISPLD2_ID
]

display(
    crispld2_result[
        [
            "gene",
            "ensembl_id",
            "untreated_mean",
            "dex_mean",
            "log2FC",
            "pvalue",
            "padj",
            "direction"
        ]
    ]
)

top_candidates_reset = top_candidates.reset_index(
    drop=True
)

match = top_candidates_reset.index[
    top_candidates_reset["ensembl_id"]
    == CRISPLD2_ID
].tolist()

if match:
    print(
        "CRISPLD2 rank among Dex-induced candidates:",
        match[0] + 1
    )
else:
    print(
        "CRISPLD2 was not included in the displayed Dex-induced candidate list."
    )


### 確認してください

- log2FCは正ですか、負ですか？
- Dexで上昇していますか？
- adjusted p-valueは0.05未満ですか？
- 自分が選んだ3候補にCRISPLD2は入っていましたか？

答え：

-
-
-
-


---
# 18. CRISPLD2の発現をdonorごとに見る

平均値だけでなく、4人のdonorそれぞれで発現変化を確認します。

paired designなので、

> **同じdonorのUntreatedとDexを線で結ぶ**

と変化が見やすくなります。


In [ ]:
crispld2_expr = (
    log_cpm
    .loc[CRISPLD2_ID]
    .rename("expression")
    .reset_index()
    .rename(columns={"index": "sample"})
    .merge(
        metadata[
            ["sample", "cell", "condition"]
        ],
        on="sample",
        how="left"
    )
)

display(crispld2_expr)


In [ ]:
plt.figure(figsize=(7, 5))

sns.stripplot(
    data=crispld2_expr,
    x="condition",
    y="expression",
    hue="cell",
    s=10
)

# donorごとに線を結ぶ
for donor, donor_df in crispld2_expr.groupby("cell"):
    donor_df = donor_df.sort_values("condition")

    plt.plot(
        donor_df["condition"].astype(str),
        donor_df["expression"],
        alpha=0.6
    )

plt.ylabel("log2(CPM + 1)")
plt.title("CRISPLD2 expression")
plt.legend(
    title="Donor",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()


### 考えてみよう

1. 4人のdonorすべてで同じ方向に変化していますか？
2. 平均値だけを見るより、paired plotは何を教えてくれますか？
3. サンプル数が4 donorだけであることには、どのような限界がありますか？

答え：

1.
2.
3.


---
# 19. 自分なら次にどんなwet実験をするか？

RNA-seqで候補遺伝子が見つかっても、

> **RNA-seqで差があった = そのgeneが重要な原因である**

とは限りません。

ここからwet研究につなげます。

例えば、

- qPCRでmRNA発現を再確認する
- Western blotやELISAでprotein levelを確認する
- siRNA / CRISPRでgeneを抑制する
- overexpressionを行う
- downstream phenotypeを見る
- cytokine産生を見る
- 別のdonor / 別のcell typeで再現性を確認する

などが考えられます。

### あなたの実験計画

#### 注目するgene

Gene：

#### 仮説

このgeneは、Dex処理によって：

#### 次に行う実験

1.
2.
3.

#### その実験で何がわかるか


---
# 20. ミニ研究レポート

最後に、この演習を短くまとめてください。

## 研究目的

Dex処理によって：

## PCAからわかったこと

## DEG解析からわかったこと

## 自分が選んだ候補遺伝子

1.
2.
3.

## 論文の主要候補CRISPLD2について、自分の解析で確認できたこと

## 次に行いたいwet実験

## この演習で一番面白かったこと


---
# 21. まとめ

今日の午後は、公開RNA-seqデータを使って、

1. count matrixとmetadataを確認する
2. 低発現遺伝子を除外する
3. CPMとlog変換を行う
4. PCAでsample全体の関係を見る
5. paired designを確認する
6. DEG解析を行う
7. 多重検定補正を行う
8. Volcano plotを作る
9. ヒートマップを見る
10. 自分で候補遺伝子を選ぶ
11. 遺伝子機能を調べる
12. 論文のCRISPLD2と答え合わせする
13. 次のwet実験を考える

という流れを体験しました。

---

## 今日一番伝えたいこと

RNA-seq解析は、

> 図を作って終わるもの

ではありません。

データ解析は、

> **候補を見つける → 仮説を立てる → 次の実験を考える**

ための研究プロセスの一部です。

公開データを使えば、自分の研究テーマについても、

- 先行研究の再解析
- 新しい仮説の探索
- 実験計画のための予備解析

を行うことができます。

---

## 参考情報

- GEO accession: GSE52778
- Himes BE et al.
- PLOS ONE. 2014;9(6):e99625
- DOI: 10.1371/journal.pone.0099625
- Bioconductor airway dataset
